### Data Sourcing

In [ ]:
import pandas as pd
import numpy as np
import os

# Define explicit relative paths
path_apc = "../data/raw/Ateneo Policy Center Philippine Political Dynasties Dataset (2022 Update).xlsx"
path_out = "../data/processed/apc_dynasty_long.csv"

# Load the specific sheet needed for the provincial target variable
print("Loading APC data...")
df_apc_raw = pd.read_excel(path_apc, sheet_name="Data - Province")

### Data Filtering

In [ ]:
target_cols = ['province'] + [col for col in df_apc_raw.columns if 'fatdynshare' in col.lower()]
df_apc_filtered = df_apc_raw[target_cols].copy()

### Dataframe Reshaping

In [ ]:
# Melt the wide format into a long format
df_melted = df_apc_filtered.melt(
    id_vars=['province'],
    var_name='year_raw',
    value_name='dynasty_share'
)

### Time-Series Formatting

In [ ]:
# Extract numeric year from strings using regex
df_melted['year'] = df_melted['year_raw'].str.extract(r'(\d{4})').astype(int)

# Rename 'province' to 'prov_name' to match Poverty and IRA dataframes
df_melted = df_melted.rename(columns={'province': 'prov_name'})

# Clean string whitespace and ensure numeric types
df_melted['prov_name'] = df_melted['prov_name'].astype(str).str.strip()
df_melted['dynasty_share'] = pd.to_numeric(df_melted['dynasty_share'], errors='coerce')

# Drop old wide column headers and sort for a clean time-series panel
df_ts = df_melted[['prov_name', 'year', 'dynasty_share']]
df_ts = df_ts.sort_values(by=['prov_name', 'year']).reset_index(drop=True)

# Final validation checks
assert pd.api.types.is_integer_dtype(df_ts['year'])
assert pd.api.types.is_numeric_dtype(df_ts['dynasty_share'])

### Feature Engineering

In [ ]:
print("Loading Politicians data to extract Executive features...")
df_pols = pd.read_excel(path_apc, sheet_name="Data - Politicians")

# Clean up the position column for exact matching (strip whitespace, uppercase)
df_pols['position_upper'] = df_pols['position'].astype(str).str.strip().str.upper()

# Filter specifically for the exact values
target_positions = ['GOVERNOR', 'VICE GOVERNOR']
df_execs = df_pols[df_pols['position_upper'].isin(target_positions)].copy()

# Quick diagnostic print
print(f"Diagnostic: Found {len(df_execs)} executive rows before pivoting.")

# Map the exact positions to our new column names using a dictionary
position_mapping = {
    'GOVERNOR': 'gov_is_dynasty',
    'VICE GOVERNOR': 'vice_gov_is_dynasty'
}
df_execs['position_clean'] = df_execs['position_upper'].map(position_mapping)

# Convert text to binary 1/0
col_dynasty_indicator = 'fat_dynasty_indicator' 

df_execs['is_fat_numeric'] = np.where(
    df_execs[col_dynasty_indicator].astype(str).str.strip().str.lower() == 'fat',
    1,
    0
)

# Pivot the data using the numeric column
df_exec_feat = df_execs.pivot_table(
    index=['province', 'year'],
    columns='position_clean',
    values='is_fat_numeric',  
    aggfunc='max' 
).reset_index()

# Guarantee both columns exist to prevent KeyErrors
for col in ['gov_is_dynasty', 'vice_gov_is_dynasty']:
    if col not in df_exec_feat.columns:
        df_exec_feat[col] = 0

# Standardize the province name to match df_ts
df_exec_feat = df_exec_feat.rename(columns={'province': 'prov_name'})
df_exec_feat['prov_name'] = df_exec_feat['prov_name'].astype(str).str.strip()

# Merge into the main time-series dataframe
df_ts = df_ts.merge(df_exec_feat, on=['prov_name', 'year'], how='left')

# Fill NaNs with 0
df_ts['gov_is_dynasty'] = df_ts['gov_is_dynasty'].fillna(0).astype(int)
df_ts['vice_gov_is_dynasty'] = df_ts['vice_gov_is_dynasty'].fillna(0).astype(int)

print("Governor and Vice Governor features successfully engineered and merged!")

### APC Dataframe Exporting

In [ ]:
os.makedirs(os.path.dirname(path_out), exist_ok=True)
df_ts.to_csv(path_out, index=False)

print(f"Successfully processed and saved {len(df_ts)} rows.")
print(f"Year range: {df_ts['year'].min()} - {df_ts['year'].max()}")
print(f"Saved to: {path_out}")
display(df_ts.head())